# 8 (Bonus): Mixture Models and the EM Algorithm

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [What is a Mixture Model?](#4.-What-is-a-Mixture-Model?)
   - 4.1 [Motivation: When One Distribution is Not Enough](#4.1-Motivation:-When-One-Distribution-is-Not-Enough)
   - 4.2 [The Mixture Model Framework](#4.2-The-Mixture-Model-Framework)
   - 4.3 [Hard vs Soft Assignments](#4.3-Hard-vs-Soft-Assignments)
5. [Gaussian Mixture Models (GMMs)](#5.-Gaussian-Mixture-Models-(GMMs))
   - 5.1 [The Multivariate Gaussian Distribution](#5.1-The-Multivariate-Gaussian-Distribution)
   - 5.2 [The GMM Density and Its Parameters](#5.2-The-GMM-Density-and-Its-Parameters)
   - 5.3 [Generating Data from a Known GMM](#5.3-Generating-Data-from-a-Known-GMM)
   - 5.4 [Covariance Types in scikit-learn](#5.4-Covariance-Types-in-scikit-learn)
6. [Training a GMM: the EM Algorithm](#6.-Training-a-GMM:-the-EM-Algorithm)
   - 6.1 [Why Not Just Use Maximum Likelihood Directly?](#6.1-Why-Not-Just-Use-Maximum-Likelihood-Directly?)
   - 6.2 [The E-Step: Computing Responsibilities](#6.2-The-E-Step:-Computing-Responsibilities)
   - 6.3 [The M-Step: Updating Parameters](#6.3-The-M-Step:-Updating-Parameters)
   - 6.4 [GMM with scikit-learn](#6.4-GMM-with-scikit-learn)
   - 6.5 [Hard vs Soft Assignments Visualised](#6.5-Hard-vs-Soft-Assignments-Visualised)
7. [Model Selection: BIC and AIC](#7.-Model-Selection:-BIC-and-AIC)
   - 7.1 [The Problem with Choosing K](#7.1-The-Problem-with-Choosing-K)
   - 7.2 [AIC and BIC](#7.2-AIC-and-BIC)
8. [GMM as a Density Estimator](#8.-GMM-as-a-Density-Estimator)
9. [Strengths, Weaknesses, and When to Use GMMs](#9.-Strengths,-Weaknesses,-and-When-to-Use-GMMs)
10. [Summary](#10.-Summary)
11. [References](#11.-References)

---

## 1. Introduction

Most clustering tutorials cover k-nearest neighbours, K-Means, hierarchical clustering, and DBSCAN. Those algorithms work well in many situations, but they all make a simplifying assumption: each data point belongs entirely to one cluster. That assumption is often unrealistic.

This notebook introduces a fourth family of clustering methods: **Mixture Models**, with a focus on **Gaussian Mixture Models (GMMs)**. The central idea is that instead of assigning each data point to exactly one cluster, we assign each point a *probability* of belonging to each cluster. This approach is more honest about the fact that cluster boundaries are rarely sharp in real data.

GMMs are also more mathematically demanding than simpler clustering algorithms, which is why they are treated separately here. The reward for that extra effort is significant: GMMs provide a full probabilistic account of the data, and every point has a probability of belonging to every component. The model also gives you the complete density function $P(\mathbf{x})$, which describes the probability distribution over all possible values of the random variable $\mathbf{x}$, allowing you to compute the likelihood of any specific outcome or range of outcomes. This opens up applications in anomaly detection and generative modelling that hard-assignment algorithms cannot support.

Investigating GMMs also leads us to the **Expectation-Maximisation (EM) algorithm**, one of the most important and widely used algorithms in machine learning. EM is a general technique for fitting models that involve hidden (unobserved) variables, and it appears in many areas beyond clustering.

### Learning Objectives

By the end of this notebook you should be able to:

- Explain what a mixture model is and why it is needed when data comes from multiple underlying distributions.
- Articulate the distinction between hard and soft cluster assignments and explain why soft assignments are more statistically honest.
- Describe the parameters of a Gaussian Mixture Model ($\boldsymbol{\mu}_k$, $\boldsymbol{\Sigma}_k$, $\pi_k$) and explain what each controls.
- Explain the intuition behind the EM algorithm and trace through the E-step and M-step equations.
- Fit GMMs using scikit-learn with different covariance structures, and interpret the fitted parameters.
- Visualise soft assignment probabilities and identify uncertain points.
- Use BIC and AIC to select the number of components, and explain the trade-off each criterion makes.
- Use a GMM as a density estimator and describe how this differs from using it as a clustering tool.

---


---

## 2. Useful Links

| Link | Description |
|------|-------------|
| [scikit-learn: mixture models](https://scikit-learn.org/stable/modules/mixture.html) | Official documentation for `GaussianMixture` and `BayesianGaussianMixture`, including the covariance type options and the full list of fitted attributes. |
| [scikit-learn: GaussianMixture API](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html) | Full API reference for `GaussianMixture`, including all constructor parameters, methods (`fit`, `predict`, `predict_proba`, `score`), and attributes (`means_`, `covariances_`, `weights_`). |
| [Wikipedia: EM algorithm](https://en.wikipedia.org/wiki/Expectation%E2%80%93maximization_algorithm) | A clear overview of the EM algorithm with derivations, convergence proofs, and links to related methods. |
| [Matplotlib: Ellipse patch](https://matplotlib.org/stable/api/patches_api.html#matplotlib.patches.Ellipse) | Documentation for the `Ellipse` patch used to draw covariance ellipses on GMM plots. |


---

## 3. Libraries and Environment Setup

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

### 3.2 Libraries Used in This Notebook

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Numerical arrays, linear algebra, and random number generation. Used throughout for matrix operations and density calculations. | [numpy.org](https://numpy.org/doc/stable/) |
| **Matplotlib** (`matplotlib.pyplot`) | All scatter plots, line plots, contour plots, and ellipse overlays. | [matplotlib.org](https://matplotlib.org/stable/) |
| **Seaborn** (`seaborn`) | Global visual theme for consistent plot styling. | [seaborn.pydata.org](https://seaborn.pydata.org/) |
| **SciPy** (`scipy.stats`) | Used for the `multivariate_normal` probability density function. | [scipy.org](https://docs.scipy.org/doc/scipy/) |
| **scikit-learn** (`sklearn`) | Provides `GaussianMixture`, dataset generators (`make_blobs`, `make_moons`), and `KMeans` for comparison. | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **ipywidgets** (`ipywidgets`) | Adds interactive UI controls (sliders, dropdowns, etc.) to Jupyter notebooks. We use it to create sliders for adjusting plot parameters in real time. | [ipywidgets.readthedocs.io](https://ipywidgets.readthedocs.io/en/stable/) |
| **ipympl** (`matplotlib widget` backend) | Enables interactive, live-updating Matplotlib figures inside Jupyter. Required for smooth slider updates without redrawing the entire plot. | [matplotlib.org/ipympl](https://matplotlib.org/ipympl/) |

### 3.3 Imports

All library imports are placed in the single cell below. Run this cell before executing anything else in the notebook.

> **You must run the cell below first.** All subsequent cells depend on the libraries and variables defined here.

In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np                          # Numerical arrays and linear algebra
import matplotlib.pyplot as plt             # Plotting and visualisation
import matplotlib.cm as cm                  # Colour maps for multi-component plots
from matplotlib.patches import Ellipse      # Used to draw Gaussian component ellipses

import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

import seaborn as sns                       # Global plot theme

from scipy.stats import multivariate_normal  # Multivariate Gaussian PDF

from sklearn.mixture import GaussianMixture  # GMM via EM algorithm
from sklearn.cluster import KMeans           # For comparison with GMM
from sklearn.datasets import make_blobs, make_moons  # Synthetic dataset generators

# A single seeded random number generator used throughout for reproducibility
rng = np.random.default_rng(42)

# Apply a consistent visual style to all figures in this notebook
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 100

# Confirm successful import
print('Libraries loaded successfully.')


---


## 4. What is a Mixture Model?

🔙 [Back to Table of Contents](#table-of-contents)

### 4.1 Motivation: When One Distribution is Not Enough

A **probability distribution** is a mathematical description of how likely different values are. A common choice is the Gaussian (Normal) distribution, which produces the familiar bell-curve shape. When we say data follows a Gaussian, we mean it is centred around a single mean value and spreads out symmetrically from there, producing that bell-curve shape when visualised. Fitting a Gaussian to data means finding the mean and variance of a Gaussian distribution that best describes the observed data.

But many real-world datasets are not well described by a single bell curve. They come from multiple distinct sub-populations, each with its own statistical character. Here are some everyday examples:

- **Customer spending patterns.** One group of customers makes frequent small purchases; another makes rare but very large ones. No single Gaussian will fit both groups at once.
- **Medical measurements.** A patient population might contain healthy individuals alongside people with a particular condition. Each group produces measurements from a different distribution.
- **Galaxy velocities.** In astronomy, the velocities of stars in a galaxy cluster are well modelled as a mixture of Gaussians, each corresponding to a different physical sub-structure.

When you fit a single Gaussian to data like this, the result is a model that sits somewhere between the sub-populations and fits none of them well. The mean of the fitted Gaussian ends up in a region where hardly any data actually lives. The **mixture model** framework addresses this by explicitly modelling the data as arising from $K$ distinct component distributions, one per sub-population.

The cell below illustrates the problem concretely. We generate data from two separate Gaussians and then try to fit a single Gaussian to all of it. The mismatch is obvious.

In [ ]:
# Listing 2.
%matplotlib widget
from visualisations.Figure_1 import show
show()

---

### 4.2 The Mixture Model Framework

A **mixture model** is a probabilistic model that represents the data as a weighted combination of $K$ simpler distributions, called **components**. Rather than assuming all observations come from a single distribution, a mixture model says: the data contains $K$ distinct sub-groups, each described by its own distribution, and each sub-group accounts for a certain proportion of the overall population.

The overall probability of observing a particular value $\mathbf{x}$ is obtained by asking: how likely is this observation under each component, weighted by how large that component is?

$$P(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, p_k(\mathbf{x} \mid \boldsymbol{\theta}_k)$$

This is a weighted sum across all $K$ components. For each component $k$, two quantities are multiplied together and then added to the total:

**The mixing coefficient $\pi_k$** is the weight assigned to component $k$. It represents the proportion of the overall population that belongs to sub-group $k$, or equivalently, the probability that a randomly chosen observation came from that sub-group. Returning to the bi-modal example in Figure 1, if 200 of 350 total observations came from group 1, then $\pi_1 = 200/350 \approx 0.57$. All mixing coefficients must be non-negative and must sum to exactly one, since together they account for the entire population:

$$\pi_k \geq 0 \quad \text{and} \quad \sum_{k=1}^{K} \pi_k = 1$$

**The component density $p_k(\mathbf{x} \mid \boldsymbol{\theta}_k)$** is the probability density function of component $k$, parameterised by its own set of parameters $\boldsymbol{\theta}_k$. For a Gaussian component, $\boldsymbol{\theta}_k$ would be the mean $\mu_k$ and standard deviation $\sigma_k$ of that sub-group. The density function answers the question: assuming this observation came from component $k$, how likely is this particular value of $\mathbf{x}$?

The overall density $P(\mathbf{x})$ is therefore the answer to a different question: marginalising over all possible components, how likely is this observation overall? A value of $\mathbf{x}$ that falls near the centre of a large, well-fitting component will have a high $P(\mathbf{x})$; a value that falls far from all components will have a low $P(\mathbf{x})$.

> 💡 **Intuition:** think of the mixture model as a weighted panel of experts. Each expert (component) has their own opinion about how likely a given observation is, expressed as a density value. The mixing coefficient is how much authority each expert is given. The final probability is the authority-weighted average of all the experts' opinions.

The **generative story** of a mixture model offers a concrete way to think about what the model assumes about how the data was produced. Imagine each data point was created in two steps: first, flip a weighted $K$-sided coin to select a component $k$, where the probability of landing on side $k$ equals $\pi_k$; then draw a sample $\mathbf{x}$ from the chosen component distribution $p_k(\mathbf{x} \mid \boldsymbol{\theta}_k)$ to produce the actual observed value. The data we see is the result of that second step only. The coin flip happened, but the outcome was never recorded. It is a **latent variable**, a hidden quantity that influenced the data but was not observed, and this hidden structure is precisely the challenge that makes fitting a mixture model more complex than fitting a single distribution. We must simultaneously estimate both the component parameters $\boldsymbol{\theta}_k$ and the mixing coefficients $\pi_k$, while never directly observing which component was responsible for each observation. The algorithm designed to do exactly this is the Expectation-Maximisation (EM) algorithm, introduced later.

### 4.3 Hard vs Soft Assignments

Before going further into GMMs, it is worth clarifying the most important philosophical difference between mixture models and simpler clustering algorithms like K-Means.

**Hard assignment** is what K-Means does. Each data point is assigned to exactly one cluster, with no ambiguity. The assignment variable $z_{ik}$ is binary: $z_{ik} = 1$ if point $i$ belongs to cluster $k$, and $z_{ik} = 0$ otherwise. There is no middle ground.

**Soft assignment** is what mixture models do. Each data point has a probability of belonging to each component. The assignment variable $r_{ik}$ is a real number between 0 and 1, called the **responsibility** of component $k$ for point $i$. The responsibilities across all components must sum to one: $\sum_k r_{ik} = 1$.

Why does this matter? Consider a point sitting exactly on the boundary between two clusters. K-Means assigns it to one cluster with complete certainty, even though that certainty is entirely artificial. A mixture model assigns it roughly equal probability to each component, which is the more honest description of a genuinely ambiguous situation.

The cell below visualises this contrast on a dataset where two clusters overlap. The third panel highlights in gold the points where the GMM is genuinely uncertain. These are exactly the points where a hard assignment would be most misleading.

In [ ]:
# Listing 3.
%matplotlib widget
from visualisations.Figure_2 import show
show()


---


## 5. Gaussian Mixture Models (GMMs)

🔙 [Back to Table of Contents](#table-of-contents)

### 5.1 The Multivariate Gaussian Distribution

A **Gaussian distribution** (also called a Normal distribution) is the classic bell-curve probability distribution. When working with data that has more than one feature, height and weight measured together, for instance, we need a generalisation called the **multivariate Gaussian**. It describes how a collection of features varies together, capturing not just how spread out each feature is, but also how pairs of features are correlated with one another.

For a $d$-dimensional data point $\mathbf{x}$ (meaning $\mathbf{x}$ has $d$ features), the multivariate Gaussian probability density is:

$$
\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma}) = \frac{1}{(2\pi)^{d/2} |\boldsymbol{\Sigma}|^{1/2}} \exp\!\left( -\frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu}) \right)
$$

There are three key quantities:

- $\boldsymbol{\mu} \in \mathbb{R}^d$ is the **mean vector**. This is the centre of the distribution, the average value in each feature dimension. If you draw many points from this Gaussian, their average will be close to $\boldsymbol{\mu}$.
- $\boldsymbol{\Sigma} \in \mathbb{R}^{d \times d}$ is the **covariance matrix**, a square $d \times d$ matrix that describes how the $d$ features spread out and relate to each other (also covered in Notebook 9). For a two-feature distribution, it looks like this:

$$\boldsymbol{\Sigma} = \begin{pmatrix} \sigma_1^2 & \sigma_{12} \\ \sigma_{12} & \sigma_2^2 \end{pmatrix}$$

The diagonal entries $\sigma_1^2$ and $\sigma_2^2$ are the **variances** of each feature individually: larger values mean that feature spreads more widely around its mean. The off-diagonal entries $\sigma_{12}$ capture the **covariance** between the two features: if they tend to increase together, $\sigma_{12}$ is positive; if one tends to decrease when the other increases, $\sigma_{12}$ is negative; if they are unrelated, $\sigma_{12}$ is zero and the matrix is diagonal. The covariance matrix is always symmetric ($\sigma_{12}$ appears in both off-diagonal positions) because the relationship between feature $i$ and feature $j$ is the same as the relationship between feature $j$ and feature $i$. In higher dimensions the same structure extends naturally: a $d \times d$ covariance matrix has $d$ variance terms on the diagonal and $d(d-1)/2$ unique covariance terms in the off-diagonal positions.

- $|\boldsymbol{\Sigma}|$ denotes the **determinant** of $\boldsymbol{\Sigma}$. Think of it as a single number that measures how spread out the distribution is overall. If the distribution is wide and flat in all directions, the determinant is large. If it is narrow and concentrated into a small region, the determinant is small. It appears in the denominator of the formula to keep the total probability summing to 1: a broader distribution must be shorter to compensate for covering more area, and dividing by $|\boldsymbol{\Sigma}|$ enforces exactly that.

The expression $(\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})$ is called the **Mahalanobis distance** from $\mathbf{x}$ to $\boldsymbol{\mu}$. It is a generalised measure of distance that accounts for the scale and orientation of the distribution. Points that are far from the mean in this sense receive low density; points close to the mean receive high density.

The shape of a multivariate Gaussian, when plotted in 2-D, is always an **ellipse**. The covariance matrix $\boldsymbol{\Sigma}$ determines everything about that ellipse: how wide it is, how tall it is, and which direction it is tilted. Changing a single entry in the covariance matrix can rotate the ellipse, stretch it, or squash it, and Figure 3 below makes this concrete by showing four representative cases side by side.

---

The four panels all share the same mean (the origin), so the only thing changing between them is the covariance matrix. In the first panel the distribution is perfectly circular, because the variance is the same in both directions and the two features are uncorrelated. The second panel stretches the ellipse along the horizontal axis by increasing the variance of $x_1$ relative to $x_2$, producing an axis-aligned oval. The third and fourth panels introduce correlation between $x_1$ and $x_2$ by adding non-zero off-diagonal entries to $\boldsymbol{\Sigma}$: a positive correlation tilts the ellipse toward the upper-right (the two features tend to increase together), while a negative correlation tilts it toward the upper-left (when one increases, the other tends to decrease).

Each panel shows two ellipses: 

* a solid one at 1 standard deviation from the mean.
* a dashed one at 2 standard deviations.

These mark the boundaries within which roughly 39% and 86% of the probability mass falls respectively for a 2-D Gaussian. The small numbers printed at the bottom of each panel are the eigenvalues of the covariance matrix, which you do not need to understand in detail yet (they are covered properly in the notebook on PCA). For now, it is enough to know that larger eigenvalues correspond to more spread in that direction, and the ratio between them determines how elongated the ellipse is.

In [ ]:
# Listing 4.
%matplotlib widget
from visualisations.Figure_3 import show
show()

### 5.2 The GMM Density and Its Parameters

A **Gaussian Mixture Model** is a mixture model (Section 4.2) where each component is a multivariate Gaussian. With $K$ components, the GMM density is:

$$
P(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)
$$

Each component $k$ has three parameters that must be estimated from the data:

- $\boldsymbol{\mu}_k \in \mathbb{R}^d$: the **mean vector** of component $k$. This is the centre of that component's cluster in feature space.
- $\boldsymbol{\Sigma}_k \in \mathbb{R}^{d \times d}$: the **covariance matrix** of component $k$. This controls the shape and orientation of the cluster's elliptical contours.
- $\pi_k \in (0, 1)$: the **mixing weight** of component $k$. This controls the relative size or importance of the cluster. A component with a large mixing weight contributes more to the overall density and tends to claim more data points.

The total number of free parameters grows quickly as $K$ and $d$ increase. For a full covariance matrix in $d$ dimensions, each component needs $d(d+1)/2$ covariance parameters (because the matrix is symmetric, so only its upper triangle is independent). Adding the $d$ mean parameters and the mixing weight (minus one, because the weights must sum to one), the full parameter count for a $K$-component GMM in $d$ dimensions is:

$$
K \cdot \frac{d(d+1)}{2} + K \cdot d + (K - 1)
$$

This growth is why high-dimensional GMMs need large datasets: you cannot estimate many parameters reliably without enough data. This is the same curse of dimensionality discussed in Notebook 4.

**An analogy.** Imagine coins arriving from $K$ different mints. Each mint produces coins with a slightly different weight distribution (its own Gaussian). The mixing coefficient $\pi_k$ tells you what fraction of all coins came from mint $k$. When you pick up a random coin and weigh it, you cannot be certain which mint made it, but you can compute a probability for each mint based on the weight. That probability is the **responsibility** of each component, which is the central quantity the EM algorithm computes.

### 5.3 Generating Data from a Known GMM

The clearest way to understand what a GMM represents is to generate data from one where we know all the parameters in advance. This is called working with a **ground-truth** model: we define the components ourselves rather than estimating them from data, which means we can compare what the model says against what we know to be true.

In scikit-learn, a GMM is represented by a `GaussianMixture` object. Normally you would call `gmm.fit(X)` to estimate the parameters from data, but you can also set the parameters directly and skip fitting entirely. The three attributes that define a GMM are:

- `means_`: a 2-D array of shape `(n_components, n_features)`, where each row is the mean of one component.
- `covariances_`: an array of shape `(n_components, n_features, n_features)`, where each entry is the covariance matrix of one component.
- `weights_`: a 1-D array of length `n_components` containing the mixing coefficients, which must sum to 1.

Once these are set, `gmm.sample(n)` generates `n` new data points by following the generative story from Section 4.2 automatically: it draws a component index from the mixing weights for each point, then samples from that component's Gaussian. It returns two arrays: the data points `X` of shape `(n, n_features)`, and `component_ids` of shape `(n,)` recording which component generated each point. Because we generated the data ourselves, we have access to `component_ids`, something that would not be available with real observed data and is precisely what a fitted GMM tries to recover.

One technical detail: scikit-learn requires a fourth internal attribute, `precisions_chol_`, to be set before `sample()` can be called. This is a pre-computed quantity derived from the covariance matrices that the model uses internally for efficient computation. The helper function `_compute_precision_cholesky` calculates it from the covariances, so you do not need to compute it by hand.

---

The code below constructs a three-component GMM with parameters we have chosen deliberately to produce an interesting and clearly structured dataset. The three components have different means (placing them in different regions of the 2-D space), different covariance matrices (giving each component a distinct shape and orientation), and different mixing weights (so the components contribute different numbers of points to the final dataset). With weights of 0.4, 0.35, and 0.25, component 0 should generate roughly twice as many points as component 2 on average, and the printed summary confirms this.

The two panels tell a before-and-after story. The left panel shows only the raw data with no labels: this is the information available when fitting a GMM to real data. Looking at it, you can probably see that the points do not form one homogeneous cloud, there appear to be two or three clusters, but it is not obvious exactly where the boundaries are or how many components are present. The right panel reveals the **ground truth** by colouring each point according to the component that generated it. The overlap between the coloured groups, particularly in the regions where two components are close together, illustrates why cluster membership is genuinely uncertain for some points and why a soft assignment model like a GMM is more appropriate than a hard assignment model like K-Means.

This ground-truth view is a luxury we only have because we generated the data ourselves. When a GMM is fitted to real data, the model must recover something like the right panel using only the information in the left panel. How it does that is the subject of the next section.

In [ ]:
# Listing 5.
# ── Define a GMM with known parameters ───────────────────────────────────────
# We construct a GaussianMixture object and set its parameters directly,
# rather than fitting them from data. This lets us sample from a model
# where we know the ground truth exactly.
gmm_true = GaussianMixture(n_components=3, covariance_type='full')

# These attributes must be set before calling sample().
# means_: one row per component, shape (n_components, n_features)
# covariances_: one matrix per component, shape (n_components, n_features, n_features)
# weights_: mixing coefficients, must sum to 1
gmm_true.means_ = np.array([
    [-3.0, -2.0],
    [ 2.0,  3.0],
    [ 5.0, -1.0],
])
gmm_true.covariances_ = np.array([
    [[1.0,  0.5], [ 0.5, 0.8]],   # positive correlation
    [[1.5, -0.6], [-0.6, 1.0]],   # negative correlation
    [[0.8,  0.0], [ 0.0, 2.0]],   # elongated vertically
])
gmm_true.weights_ = np.array([0.4, 0.35, 0.25])

# sklearn requires precisions_chol_ to be set before sampling.
# compute_precision_cholesky computes this from the covariances internally.
from sklearn.mixture._gaussian_mixture import _compute_precision_cholesky
gmm_true.precisions_chol_ = _compute_precision_cholesky(
    gmm_true.covariances_, 'full'
)

# ── Sample from the model ─────────────────────────────────────────────────────
# sample() returns (X, component_ids): the data points and the index of
# the component that generated each one, following the generative story.
X_gmm, component_ids = gmm_true.sample(500)

print(f'Generated {len(X_gmm)} samples from a 3-component GMM')
for k in range(3):
    n_k = (component_ids == k).sum()
    print(f'  Component {k}: weight = {gmm_true.weights_[k]:.2f}, '
          f'samples generated = {n_k}')

# ── Plot ──────────────────────────────────────────────────────────────────────
palette = ['steelblue', 'tomato', 'seagreen']

fig, axes = plt.subplots(1, 2, figsize=(8, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# Left: what we would observe without labels
axes[0].scatter(X_gmm[:, 0], X_gmm[:, 1],
                color='lightsteelblue', s=20, alpha=0.7,
                edgecolors='k', lw=0.1)
axes[0].set_title('What we observe\n(no component labels)', fontsize=10)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.25)

# Right: ground-truth labels, only possible because we generated the data
for k, col in enumerate(palette):
    mask = component_ids == k
    axes[1].scatter(X_gmm[mask, 0], X_gmm[mask, 1],
                    color=col, s=20, alpha=0.7, edgecolors='k', lw=0.1,
                    label=f'Component {k}  (w={gmm_true.weights_[k]:.2f}, n={mask.sum()})')
axes[1].set_title('Ground truth\n(which component generated each point)', fontsize=10)
axes[1].set_xlabel('Feature 1')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.25)

plt.suptitle('Figure 4: Sampling from a known 3-component GMM', fontsize=12)
plt.tight_layout()
plt.show()

---

### 5.4 Covariance Types in scikit-learn

In Section 5.3 we generated data from a GMM where we knew the covariance matrices in advance and set them directly. In practice, we never know the true covariance matrices: we only have the observed data, and we must estimate everything, including $\boldsymbol{\mu}_k$, $\pi_k$, and $\boldsymbol{\Sigma}_k$ for every component, from that data alone. The covariance matrix is therefore not a given; it is one of the things the fitting process is trying to recover.

This matters because the covariance matrix is the most parameter-hungry part of the model. A mean vector in $d$ dimensions requires $d$ numbers. A full covariance matrix requires $d(d+1)/2$ numbers, which grows much faster: in 10 dimensions that is 55 parameters per component just for the covariance, and in 100 dimensions it is 5,050. Estimating that many parameters reliably requires a large amount of data, and if the dataset is small relative to the number of parameters, the model is likely to overfit.

A covariance matrix can take many forms: it might produce circular clusters (equal variance in all directions, no correlation), axis-aligned ellipses (different variances per feature, no correlation), or ellipses in any orientation (full correlation structure between features). The more expressive the covariance structure, the more parameters must be estimated. Allowing each component to have a fully unconstrained covariance matrix gives the GMM maximum flexibility to fit whatever shape the data takes, but it also means estimating the largest number of parameters, which requires more data and increases the risk of overfitting in high dimensions.

scikit-learn's `GaussianMixture` provides four covariance types via the `covariance_type` parameter. They trade off flexibility against the number of parameters that must be estimated from the data:

| `covariance_type` | Shape of $\boldsymbol{\Sigma}_k$ | Parameters per component | Cluster shape |
|---|---|---|---|
| `'full'` | Unconstrained symmetric positive-definite matrix | $d(d+1)/2$ | Arbitrary ellipse, any orientation |
| `'tied'` | All components share one covariance matrix | $d(d+1)/2$ (shared once) | Same ellipse shape for all components |
| `'diag'` | Diagonal matrix (features are uncorrelated) | $d$ | Axis-aligned ellipse |
| `'spherical'` | Scalar multiple of the identity: $\sigma_k^2 \mathbf{I}$ | 1 | Circle or sphere |

The most restrictive type, `'spherical'`, is mathematically equivalent to K-Means when we take the limit of hard assignments. The most flexible, `'full'`, can capture correlations between features at the cost of many more parameters.

The basic usage pattern is the same regardless of which type you choose:

```python
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(
    n_components=3,          # number of Gaussian components K
    covariance_type='full',  # 'full', 'tied', 'diag', or 'spherical'
    n_init=5,                # number of random restarts; keeps the best result
    random_state=0,          # for reproducibility
)
gmm.fit(X)                   # X has shape (n_samples, n_features)

# After fitting, the estimated parameters are stored as attributes.
gmm.means_        # shape (n_components, n_features): estimated component means
gmm.covariances_  # shape depends on covariance_type (see below)
gmm.weights_      # shape (n_components,): estimated mixing weights, sum to 1
```

The shape of `gmm.covariances_` varies by type. For `'full'`, it is `(n_components, d, d)`, one full matrix per component. For `'tied'`, it is `(d, d)`, a single shared matrix. For `'diag'`, it is `(n_components, d)`, one variance per feature per component. For `'spherical'`, it is `(n_components,)`, one scalar variance per component.

---

The cell below fits a separate GMM for each of the four covariance types to the same dataset generated in Section 5.3. Each model has three components but a different assumption about the shape those components can take. For each fitted model, the hard cluster assignments are shown as coloured points and a 2-standard-deviation ellipse is drawn for each component using the fitted covariance matrix.

Because the covariance is stored in a different format for each type, the code converts it to a consistent full $(2 \times 2)$ matrix before drawing the ellipse: a spherical covariance is a single scalar that becomes $\sigma^2 \mathbf{I}$; a diagonal covariance is a vector of variances that becomes a diagonal matrix; a tied covariance is a single shared matrix used for all components; and a full covariance is already a per-component matrix.

Each panel also reports the **BIC** (Bayesian Information Criterion), a score that rewards a good fit to the data while penalising the number of parameters estimated. Lower BIC is better. Comparing BIC across the four panels answers a practical question: does the extra flexibility of `'full'` produce a sufficiently better fit to justify estimating many more parameters, or does a simpler type like `'diag'` achieve nearly the same result with far fewer? On this particular dataset, where the true components have full covariance structure, you should see `'full'` achieving the lowest BIC.


In [ ]:
# Listing 6

def draw_gaussian_ellipse(ax, mu, cov, n_std=2, color='steelblue', lw=2, ls='-', label=None):
    """
    Draw an ellipse representing n_std standard deviations of a 2-D Gaussian.

    The ellipse axes and orientation are derived from the eigendecomposition
    of the covariance matrix. np.linalg.eigh is used rather than eig because
    it is numerically more stable for symmetric matrices.
    """
    vals, vecs = np.linalg.eigh(cov)
    angle  = np.degrees(np.arctan2(*vecs[:, 1][::-1]))
    width  = 2 * n_std * np.sqrt(vals[1])
    height = 2 * n_std * np.sqrt(vals[0])
    ell = Ellipse(
        xy=mu, width=width, height=height, angle=angle,
        edgecolor=color, fc='none', lw=lw, linestyle=ls, label=label,
    )
    ax.add_patch(ell)
    ax.scatter(*mu, color=color, s=80, marker='+', lw=2.5, zorder=5)


# ── Covariance type comparison ────────────────────────────────────────────────
# The four covariance types ordered from most constrained (fewest parameters)
# to most flexible (most parameters).
cov_types = ['spherical', 'diag', 'tied', 'full']

cov_type_labels = {
    'spherical': 'Spherical\n(circles, 1 param/component)',
    'diag':      'Diagonal\n(axis-aligned, d params/component)',
    'tied':      'Tied\n(shared cov, d(d+1)/2 params total)',
    'full':      'Full\n(arbitrary ellipses, d(d+1)/2 params/component)',
}

fig, axes = plt.subplots(1, 4, figsize=(13, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

for ax, cov_type in zip(axes, cov_types):

    # Fit a fresh GMM for each covariance type on the same dataset X_gmm.
    # n_init=5 runs the EM algorithm from 5 different random initialisations
    # and keeps the result with the highest log-likelihood, reducing the risk
    # of converging to a poor local optimum.
    gmm_ct = GaussianMixture(
        n_components=3,
        covariance_type=cov_type,
        random_state=0,
        n_init=5,
    )
    gmm_ct.fit(X_gmm)

    # predict() returns the hard cluster label for each point, i.e. the
    # component with the highest responsibility. This is used only for
    # colouring the scatter plot; the ellipses are drawn from the fitted
    # covariance matrices directly.
    hard = gmm_ct.predict(X_gmm)

    # tab10 is a qualitative colourmap with 10 distinct colours. Passing
    # n=3 restricts it to the first three so each component gets a unique
    # colour that is consistent across panels.
    cmap_ct = plt.get_cmap('tab10', 3)

    for k in range(3):
        mask = hard == k
        ax.scatter(X_gmm[mask, 0], X_gmm[mask, 1],
                   color=cmap_ct(k), s=15, alpha=0.6,
                   edgecolors='k', lw=0.1)

    # Draw a 2-std ellipse for each fitted component. The covariance is
    # stored in different shapes depending on the covariance_type, so it
    # must be converted to a full (2, 2) matrix before passing to
    # draw_gaussian_ellipse, which always expects a square matrix.
    for k in range(3):
        mu_fit = gmm_ct.means_[k]

        if cov_type == 'spherical':
            # covariances_ is shape (n_components,): one scalar variance per
            # component. Multiply the identity matrix by that scalar to get
            # the equivalent (2, 2) matrix (a circle).
            cov_fit = np.eye(2) * gmm_ct.covariances_[k]

        elif cov_type == 'diag':
            # covariances_ is shape (n_components, d): one variance per
            # feature per component. np.diag converts the 1-D array of
            # variances into a diagonal (2, 2) matrix.
            cov_fit = np.diag(gmm_ct.covariances_[k])

        elif cov_type == 'tied':
            # covariances_ is shape (d, d): a single shared matrix for all
            # components. All ellipses will have the same shape and
            # orientation, only their centres (means) differ.
            cov_fit = gmm_ct.covariances_

        else:
            # full: covariances_ is shape (n_components, d, d).
            # Index by k to get the (2, 2) matrix for this component.
            cov_fit = gmm_ct.covariances_[k]

        draw_gaussian_ellipse(ax, mu_fit, cov_fit, n_std=2,
                              color=cmap_ct(k), lw=2.5)

    # BIC (Bayesian Information Criterion) penalises model complexity: it
    # rewards a good fit to the data but adds a penalty proportional to the
    # number of parameters estimated. Lower BIC = better balance between fit
    # and complexity. Comparing BIC across the four panels shows whether the
    # extra parameters in 'full' are justified by a sufficiently better fit.
    bic = gmm_ct.bic(X_gmm)
    ax.set_title(f'{cov_type_labels[cov_type]}\nBIC = {bic:.0f}', fontsize=9)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_xlim(-8, 10)
    ax.set_ylim(-7, 8)
    ax.grid(True, alpha=0.2)

plt.suptitle(
    'Figure 5: Effect of covariance type on fitted GMM components\n'
    '2-std ellipses shown; BIC penalises model complexity (lower = better)',
    fontsize=11,
)
plt.tight_layout()
plt.show()

---

## 6. Training a GMM: the EM Algorithm

🔙 [Back to Table of Contents](#table-of-contents)

Training a GMM means finding the set of parameters, the means, covariances, and mixing weights of each component, that best explains the observed data. In plain terms, we are asking: what collection of Gaussian blobs, and in what proportions, would most plausibly have produced the data points we see?

The challenge is that we cannot answer this directly. To estimate the parameters of each component, we need to know which points belong to it. But to know which points belong to each component, we need to know the parameters. This is the circular dependency that makes GMMs harder to fit than a single Gaussian.

The solution is an algorithm called **Expectation-Maximisation (EM)**, which breaks the circularity by alternating between two steps:

**Step 1: make a soft guess about membership.** Given the current estimate of the parameters (initially set at random), compute for every data point the probability that it came from each component. These probabilities are called **responsibilities**: they are soft, fractional memberships rather than hard labels. A point sitting squarely in the middle of one component might be assigned 95% responsibility to that component and 5% to everything else. A point in an overlapping region might be split 60/40 between two components.

**Step 2: update the parameters using those guesses.** Given the current responsibilities, recompute the best possible parameters for each component. Each component's mean shifts toward the points that most strongly belong to it, its covariance reshapes to match how those points are spread out, and its mixing weight adjusts to reflect how much of the data it accounts for overall.

These two steps are repeated in turn. After each complete cycle, the model fits the data a little better than before. The process continues until the improvement becomes negligibly small, at which point the parameters have converged.

The result is a model that has found a set of Gaussian components whose combined density matches the structure of the data, along with a soft assignment of every data point to every component. You can then obtain a hard cluster label for each point by simply taking the component with the highest responsibility, but the soft assignments are available too and carry richer information about uncertainty in ambiguous regions.

For those who want to understand the mathematical mechanics behind each step in more detail, Sections 6.1, 6.2, and 6.3 work through the log-likelihood objective, the responsibility formula, and the parameter update equations respectively. They can be read selectively or skipped entirely without losing the thread of what follows.

### 6.1 Why Not Just Use Maximum Likelihood Directly?

When we fit a GMM to data, what we are really doing is searching for the values of three sets of parameters, the means $\boldsymbol{\mu}_k$, the covariance matrices $\boldsymbol{\Sigma}_k$, and the mixing weights $\pi_k$, that best explain the observed data. "Best" here means in the maximum likelihood sense: we want the parameter values that make the data we actually observed as probable as possible under the model.

For a single Gaussian, maximum likelihood estimation (MLE) has a beautifully simple solution. The mean that maximises the likelihood is just the sample average of all the data points, and the covariance that maximises it is just the sample covariance. You can compute both in a single pass through the data and you are done.

For a GMM, the log-likelihood of the entire dataset is:

$$\ell(\boldsymbol{\theta}) = \sum_{i=1}^{n} \log \left( \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k) \right)$$

Reading this from the inside out: for each data point $\mathbf{x}_i$, we compute how likely it is under each component (weighted by the mixing coefficient), sum those likelihoods together to get the total probability of observing $\mathbf{x}_i$, take the log, and then sum across all $n$ data points. The parameters $\boldsymbol{\theta}$ we are trying to optimise are all the $\boldsymbol{\mu}_k$, $\boldsymbol{\Sigma}_k$, and $\pi_k$ simultaneously.

The problem is the **log of a sum**. In the single-Gaussian case, the log and the Gaussian density interact cleanly, and setting the derivative to zero gives a closed-form answer. With a sum inside the log, this no longer works. The equations you get when you try to set the derivative to zero are circular: the optimal mean of component $k$ depends on which data points belong to component $k$, but which data points belong to component $k$ depends on the current means of all components. There is no way to solve for one without already knowing the other.

The root cause is the **latent variable**: the hidden component identity that we discussed in the generative story in Section 4.2. If we knew which component generated each data point, the problem would collapse into something easy: just compute the weighted mean and covariance of each group of points separately, exactly as we would for a single Gaussian. But we do not know the component assignments, the whole point is that this information was not recorded, and the assignments we would infer depend on the parameters we are trying to find.

The **EM algorithm** (Expectation-Maximisation) breaks this circularity by splitting the problem into two steps that are each easy on their own and alternating between them:

1. **E step (Expectation):** take the current parameter estimates as given and use them to compute, for every data point, the probability that it came from each component. These probabilities are called **responsibilities**. They are soft assignments: instead of committing each point to one component, we give it a fractional membership in all of them.

2. **M step (Maximisation):** take the current responsibilities as given and use them to compute new, better parameter estimates. Each component's mean is updated to the responsibility-weighted average of all the data points, its covariance is updated to the responsibility-weighted covariance, and its mixing weight is updated to the average responsibility across all points.

After each complete E+M cycle, the log-likelihood is guaranteed to be at least as large as before, and in practice it strictly increases. The algorithm continues until the improvement between iterations falls below a small tolerance, at which point it has converged to a local maximum of the log-likelihood.

The key word is **local**: EM does not guarantee finding the best possible set of parameters globally. A different starting point might converge to a different local maximum with a higher log-likelihood. This is why `n_init` in scikit-learn's `GaussianMixture` runs the algorithm several times from different random starting points and returns the best result found.

### 6.2 The E-Step: Computing Responsibilities

In the **Expectation step**, we hold the current parameter estimates fixed and use them to answer a specific question for every data point: given where this point is in feature space, and given what the current components look like, how much credit should each component receive for having generated it? The answer is a number called the **responsibility** $r_{ik}$, which represents the probability that observation $i$ was generated by component $k$.

$$r_{ik} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j=1}^{K} \pi_j \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

This formula is an application of **Bayes' theorem**: we are updating our prior belief about which component generated the point (the mixing weight $\pi_k$) in light of the evidence provided by the observed value of $\mathbf{x}_i$.

Working through the formula piece by piece:

The **numerator** $\pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ asks: how probable is it that this point both came from component $k$ and has the value we observed? It is the product of two things: how common component $k$ is overall (the mixing weight $\pi_k$), and how well component $k$'s Gaussian fits this particular data point (the density $\mathcal{N}$). A large mixing weight alone is not enough to earn high responsibility if the point falls far from that component's mean; similarly, a well-placed component earns less credit if it represents only a tiny fraction of the population.

The **denominator** $\sum_{j=1}^{K} \pi_j \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)$ is the total probability of observing $\mathbf{x}_i$ under the full mixture, summing over all $K$ components. Dividing by it normalises the responsibilities so that they sum to exactly one across all components for each point:

$$\sum_{k=1}^{K} r_{ik} = 1 \quad \text{for every } i$$

This is what makes the responsibilities a proper probability distribution over components for each observation. Rather than making a hard decision about which component a point belongs to, the E-step hedges: a point in an ambiguous region between two components will have responsibilities split between them, while a point deep inside one component's territory will assign almost all responsibility to that component and nearly zero to the others.

> 💡 **Intuition:** returning to the coin analogy from Section 5.2, imagine you pick up a coin and weigh it. The weight alone does not tell you definitively which mint made it, but you can compute how likely each mint is to have produced a coin of that weight, given what you know about each mint's weight distribution and how many coins each mint produces. The responsibility is exactly that probability, computed separately for each coin and each mint.

After the E-step, we have an $n \times K$ matrix of responsibilities $\mathbf{R}$, one row per data point and one column per component. This matrix is the "soft assignment" of points to components, and it is the input to the M-step.

### 6.3 The M-Step: Updating Parameters

In the **Maximisation step**, we take the responsibilities computed in the E-step as fixed and use them to find new parameter values that maximise the expected log-likelihood. The key insight that makes EM tractable is that once the responsibilities are known, the optimal parameter updates can be written down in closed form: there is no need to search or iterate, just plug the responsibilities into three explicit formulae and compute.

The starting point is defining the **effective number of points** assigned to component $k$:

$$N_k = \sum_{i=1}^{n} r_{ik}$$

Rather than counting how many points have been hard-assigned to component $k$, $N_k$ accumulates the fractional membership of every point in the dataset. A point with responsibility 0.9 for component $k$ contributes 0.9 to $N_k$; a point with responsibility 0.05 contributes 0.05. $N_k$ is therefore a soft count, and the three parameter updates are all weighted by these soft counts.

**Updated mean:** the responsibility-weighted average position of all data points.

$$\boldsymbol{\mu}_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{n} r_{ik} \, \mathbf{x}_i$$

Points that strongly belong to component $k$ (high $r_{ik}$) pull its mean toward them; points that barely belong (low $r_{ik}$) have little influence. If all responsibilities were hard (either 0 or 1), this would reduce to the ordinary arithmetic mean of the assigned points, exactly the K-Means centroid update. The GMM M-step is a soft, weighted generalisation of that update.

**Updated covariance:** the responsibility-weighted scatter of data points around the new mean.

$$\boldsymbol{\Sigma}_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{n} r_{ik} \, (\mathbf{x}_i - \boldsymbol{\mu}_k^{\text{new}})(\mathbf{x}_i - \boldsymbol{\mu}_k^{\text{new}})^\top$$

The outer product $(\mathbf{x}_i - \boldsymbol{\mu}_k^{\text{new}})(\mathbf{x}_i - \boldsymbol{\mu}_k^{\text{new}})^\top$ is a $(d \times d)$ matrix that captures how far point $i$ is from the component mean and in which direction. Summing these matrices across all points, weighted by responsibility, gives a measure of how the points assigned to component $k$ are spread out and correlated. Again, if responsibilities were hard, this would reduce to the ordinary sample covariance of the assigned points.

**Updated mixing weight:** the fraction of total soft-assigned membership belonging to component $k$.

$$\pi_k^{\text{new}} = \frac{N_k}{n}$$

If component $k$ ends up with high total responsibility across all points, its mixing weight increases, reflecting the fact that it accounts for a larger share of the data. If it ends up with very low total responsibility, its weight shrinks, and it becomes a smaller component.

After computing all three updates for all $K$ components, the M-step is complete and the new parameters are passed back to the E-step for the next iteration.

**Convergence.** The E-step and M-step alternate until one of two stopping conditions is met. The primary condition is that the improvement in log-likelihood between successive iterations falls below a tolerance threshold, `tol=1e-3` by default in scikit-learn: at that point the parameters have stabilised and further iterations would change them only negligibly. The secondary condition is reaching the maximum number of iterations, controlled by `max_iter` (default 100), which acts as a safety limit in case convergence is slow.

A fundamental property of EM is that the log-likelihood is **guaranteed never to decrease** from one iteration to the next: each E-step and M-step together produce parameters that are at least as good as the previous ones. This does not mean the algorithm will find the globally best parameters, only that it will steadily improve until it reaches a local maximum and stops there. Running with `n_init > 1` in scikit-learn mitigates this by repeating the whole process from different random starting points and returning the best result found.

### 6.4 GMM with scikit-learn

In practice you will never implement the EM algorithm by hand: scikit-learn's `GaussianMixture` handles all of the steps described in Sections 6.1 to 6.3 internally. The code below shows the complete workflow from constructing the model to extracting predictions and evaluating the fit. Each step maps directly to a concept introduced earlier in this section.

**Step 1: construct the model.** `GaussianMixture` takes several parameters that control how EM runs. The two most important are `n_components`, which sets $K$ (the number of Gaussian components to fit), and `covariance_type`, which determines the shape each component is allowed to take (covered in Section 5.4). `n_init` runs the full EM procedure multiple times from different random starting points and keeps the best result, which reduces the risk of converging to a poor local maximum. `tol` and `max_iter` together control when the algorithm stops: it halts when the log-likelihood improvement between iterations falls below `tol`, or when `max_iter` iterations have been completed, whichever comes first.

**Step 2: fit the model.** Calling `gmm.fit(X)` runs all EM iterations internally. After fitting, `gmm.n_iter_` tells you how many iterations were needed to converge. The estimated parameters are stored as attributes with trailing underscores: `gmm.means_` contains the component means, `gmm.covariances_` contains the covariance matrices, and `gmm.weights_` contains the mixing coefficients.

**Step 3: assign points to components.** `gmm.predict(X)` returns a hard integer label for each point, the index of the component with the highest responsibility. `gmm.predict_proba(X)` returns the full responsibility matrix of shape `(n_samples, n_components)`, giving the soft probability of belonging to each component for every point. Each row of this matrix sums to 1.

**Step 4: evaluate the fit.** `gmm.score_samples(X)` returns the log-likelihood $\log P(\mathbf{x}_i)$ under the fitted model for each individual point. `gmm.score(X)` returns the mean log-likelihood per sample across the whole dataset, so multiplying by `len(X)` gives the total log-likelihood, the quantity EM was maximising throughout training.

```python
from sklearn.mixture import GaussianMixture
import numpy as np

# ── Step 1: construct the model ───────────────────────────────────────────────
gmm = GaussianMixture(
    n_components=3,          # K: the number of Gaussian components to fit
    covariance_type='full',  # shape each component can take (see Section 5.4)
    n_init=5,                # run EM from 5 random starts, keep the best result
    max_iter=100,            # maximum EM iterations per restart
    tol=1e-3,                # stop when log-likelihood improvement falls below this
    random_state=0,          # fixed seed for reproducibility
)

# ── Step 2: fit the model ─────────────────────────────────────────────────────
# X_gmm is the 2-D dataset generated in Section 5.3.
gmm.fit(X_gmm)

print(f'Converged in {gmm.n_iter_} EM iterations')
print()
print('Fitted means (one row per component):')
print(np.round(gmm.means_, 2))
print()
print('Fitted mixing weights (sum to 1):')
print(np.round(gmm.weights_, 3))

# ── Step 3: assign points to components ───────────────────────────────────────
labels = gmm.predict(X_gmm)         # hard label: index of highest-responsibility component
probs  = gmm.predict_proba(X_gmm)   # soft responsibilities: shape (n_samples, n_components)

print()
print(f'Hard labels for the first 10 points: {labels[:10]}')
print()
print('Soft responsibilities for the first 5 points (each row sums to 1):')
print(np.round(probs[:5], 3))

# ── Step 4: evaluate the fit ──────────────────────────────────────────────────
log_prob_per_sample = gmm.score_samples(X_gmm)   # log P(x_i) per point
total_log_lik       = gmm.score(X_gmm) * len(X_gmm)   # total log-likelihood

print()
print(f'Mean log-likelihood per sample : {gmm.score(X_gmm):.3f}')
print(f'Total log-likelihood           : {total_log_lik:.1f}')
```

---

In the code below we apply the workflow from Section 6.4 to a new dataset with deliberate class overlap, making the soft assignment behaviour of the GMM more visible than it would be on cleanly separated clusters.

The dataset is generated using `make_blobs` with three centres and a standard deviation of 1.8, which is large enough that the three clusters bleed into each other in the region between them. This is a more realistic scenario than the well-separated case: in real data, the boundaries between groups are rarely clean, and a model that can express uncertainty about which group a point belongs to is more honest than one that forces a hard decision.

The GMM is fitted with `n_components=3` and `covariance_type='full'`, allowing each component to take any elliptical shape. After fitting, `predict()` gives each point a hard label (the component with the highest responsibility) and `predict_proba()` gives the full responsibility matrix. Points where no single component holds more than 80% of the responsibility are flagged as uncertain: these are the points sitting in the overlap regions where the model is genuinely unsure which component generated them.

The two panels show the same fitted model from two different perspectives. The left panel uses hard labels with uncertain points highlighted in gold, making the ambiguous region immediately visible as a band between the clusters. The right panel encodes the soft responsibility directly as the opacity of each point: a point with 95% responsibility for component 0 is drawn almost fully opaque in that component's colour, while a point split 55/45 between two components appears faint, reflecting the model's uncertainty. Together the two panels illustrate the key advantage of a GMM over K-Means: rather than drawing an arbitrary hard boundary through an ambiguous region, the model acknowledges that some points genuinely could have come from more than one component.

In [ ]:
# Listing 7.

# ── Dataset ───────────────────────────────────────────────────────────────────
# Three clusters with substantial overlap (cluster_std=1.8) so that the
# soft assignments produced by the GMM are more informative than hard
# K-Means labels would be. A balanced dataset with 600 points gives each
# component enough samples to estimate a full covariance matrix reliably.
X_new, _ = make_blobs(
    n_samples=600,
    centers=[[-3, -2], [1, 3], [5, 0]],
    cluster_std=1.8,
    random_state=42,
)

# ── Fit the GMM ───────────────────────────────────────────────────────────────
gmm_new = GaussianMixture(
    n_components=3,
    covariance_type='full',
    n_init=5,
    random_state=42,
)
gmm_new.fit(X_new)

print(f'Converged in {gmm_new.n_iter_} iterations')
print()
print('Fitted means:')
print(np.round(gmm_new.means_, 2))
print()
print('Fitted mixing weights:')
print(np.round(gmm_new.weights_, 3))

# ── Assign points ─────────────────────────────────────────────────────────────
labels_new = gmm_new.predict(X_new)
probs_new  = gmm_new.predict_proba(X_new)

# Points where no component dominates (max responsibility < 0.80) are
# genuinely uncertain and are the most interesting cases to inspect.
uncertain = probs_new.max(axis=1) < 0.80
print(f'\nUncertain points (max responsibility < 0.80): '
      f'{uncertain.sum()} ({100 * uncertain.mean():.1f}%)')

# ── Evaluate ──────────────────────────────────────────────────────────────────
print(f'Mean log-likelihood per sample : {gmm_new.score(X_new):.3f}')
print(f'Total log-likelihood           : {gmm_new.score(X_new) * len(X_new):.1f}')
print(f'BIC                            : {gmm_new.bic(X_new):.1f}')

# ── Plot ──────────────────────────────────────────────────────────────────────
palette = ['steelblue', 'tomato', 'seagreen']

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# Left: hard assignments with uncertain points highlighted in gold.
ax = axes[0]
for k, col in enumerate(palette):
    mask = (labels_new == k) & ~uncertain
    ax.scatter(X_new[mask, 0], X_new[mask, 1],
               color=col, s=25, alpha=0.6, edgecolors='k', lw=0.1,
               label=f'Component {k}')
ax.scatter(X_new[uncertain, 0], X_new[uncertain, 1],
           color='gold', s=60, edgecolors='k', lw=0.6, zorder=5,
           label=f'Uncertain (n={uncertain.sum()})')
ax.set_title('Hard assignments\n(gold = uncertain)', fontsize=10)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

# Right: soft assignments shown as opacity, so the degree of uncertainty
# is visible continuously rather than as a binary threshold.
ax = axes[1]
for k, col in enumerate(palette):
    for i in np.where(labels_new == k)[0]:
        ax.scatter(X_new[i, 0], X_new[i, 1],
                   color=col, s=25,
                   alpha=float(probs_new[i, k]),   # opacity = responsibility
                   edgecolors='none')
ax.set_title('Soft assignments\n(opacity = responsibility)', fontsize=10)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid(True, alpha=0.25)

plt.suptitle('Figure 6: GMM fitted to overlapping data', fontsize=12)
plt.tight_layout()
plt.show()

### 6.5 Hard vs Soft Assignments Visualised

One of the most useful things a GMM can tell you that K-Means cannot is which data points it is uncertain about. Every point in a K-Means model receives a hard label regardless of how close it sits to a cluster boundary. A GMM instead produces a full probability distribution over components for each point, and inspecting that distribution reveals not just where the model thinks each point belongs, but how confident it is about that decision.

The key method is `predict_proba(X)`, which returns the responsibility matrix of shape `(n_samples, n_components)`. Each row is a probability distribution over the $K$ components for one data point, and the values in that row sum to 1. A point deep inside one cluster might have responsibilities of `[0.97, 0.02, 0.01]`, meaning the model is almost certain which component generated it. A point sitting in the overlap between two clusters might have `[0.52, 0.44, 0.04]`, meaning the model is genuinely unsure and a hard label would be somewhat arbitrary.

```python
# probs[i, k] is the responsibility of component k for point i.
# Each row sums to 1 across all K components.
probs = gmm.predict_proba(X)       # shape (n_samples, n_components)

# The hard label is simply the component with the highest responsibility.
# predict() is equivalent to probs.argmax(axis=1).
labels = gmm.predict(X)

# The maximum responsibility per point summarises how confident the
# assignment is: a value close to 1.0 means near-certain, a value
# close to 1/K means the point is spread evenly across all components.
max_prob = probs.max(axis=1)       # shape (n_samples,)

# Flag points where no single component dominates. The threshold of 0.75
# is a choice: lower values flag fewer points as uncertain, higher values
# flag more. There is no universally correct value.
uncertain = max_prob < 0.75
```
---

Visualising these three quantities side by side, the hard labels, the confidence of each assignment, and the uncertain points highlighted separately, makes the advantage of soft assignments concrete. The code below applies this to the GMM fitted in Section 6.4 and produces three panels showing the same data from each of these perspectives.

In [ ]:
# Listing 8.
# The responsibility matrix and hard labels from the GMM fitted in Section 6.4.
probs     = gmm_new.predict_proba(X_new)   # shape (n_samples, n_components)
labels    = gmm_new.predict(X_new)          # hard label: argmax of each row
max_prob  = probs.max(axis=1)               # confidence of each assignment
uncertain = max_prob < 0.75                 # True where no component dominates

print(f'Total points     : {len(X_new)}')
print(f'Uncertain points : {uncertain.sum()} ({100 * uncertain.mean():.1f}%)')
print()
print('Responsibility breakdown for the first 5 points:')
print('(columns = components, rows = points, each row sums to 1)')
print(np.round(probs[:5], 3))

palette = ['steelblue', 'tomato', 'seagreen']

fig, axes = plt.subplots(1, 3, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# ── Panel 1: hard assignments ─────────────────────────────────────────────────
# Every point receives the label of its highest-responsibility component,
# with no expression of uncertainty. This is what K-Means would produce.
ax = axes[0]
for k, col in enumerate(palette):
    mask = labels == k
    ax.scatter(X_new[mask, 0], X_new[mask, 1],
               color=col, s=25, alpha=0.7, edgecolors='k', lw=0.1,
               label=f'Component {k}  (n={mask.sum()})')
ax.set_title('Hard assignments\n(argmax of responsibilities)', fontsize=10)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

# ── Panel 2: soft assignments (opacity = confidence) ──────────────────────────
# Build an RGBA colour array so that each point's alpha is set to its maximum
# responsibility in a single scatter call. Each point takes its component
# colour but its opacity reflects how confident the assignment is: a point
# with max responsibility 0.99 appears nearly opaque; one with 0.52 appears
# faint. Clipping alpha to [0.05, 1.0] prevents any point from vanishing
# completely, keeping the overall structure of the data visible.
ax = axes[1]
rgba_colours = np.zeros((len(X_new), 4))
for k, col in enumerate(palette):
    mask = labels == k
    rgb = plt.matplotlib.colors.to_rgb(col)
    rgba_colours[mask, :3] = rgb
    rgba_colours[mask, 3]  = np.clip(max_prob[mask], 0.05, 1.0)

ax.scatter(X_new[:, 0], X_new[:, 1],
           c=rgba_colours, s=25, edgecolors='none')
ax.set_title('Soft assignments\n(opacity = max responsibility)', fontsize=10)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid(True, alpha=0.25)

# ── Panel 3: uncertain points highlighted ─────────────────────────────────────
# Certain points are coloured by component as in panel 1. Points below the
# responsibility threshold are overplotted in gold so the ambiguous region
# is immediately visible without needing to interpret opacity.
ax = axes[2]
for k, col in enumerate(palette):
    mask = (labels == k) & ~uncertain
    ax.scatter(X_new[mask, 0], X_new[mask, 1],
               color=col, s=25, alpha=0.6, edgecolors='k', lw=0.1,
               label=f'Component {k} (certain)')
ax.scatter(X_new[uncertain, 0], X_new[uncertain, 1],
           color='gold', s=60, edgecolors='k', lw=0.6, zorder=5,
           label=f'Uncertain (max prob < 0.75)\nn = {uncertain.sum()}')
ax.set_title('Uncertain points highlighted\n(max responsibility < 0.75)', fontsize=10)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)

plt.suptitle(
    'Figure 7: Hard vs soft GMM assignments on overlapping data\n'
    'the uncertain region is where a hard label is most misleading',
    fontsize=11,
)
plt.tight_layout()
plt.show()

At this point you have seen how to construct, fit, and interrogate a Gaussian Mixture Model using scikit-learn. Concretely, you can now:

- Define a GMM with a chosen number of components and covariance type, and fit it to a dataset using `GaussianMixture.fit()`.
- Retrieve the fitted parameters: the component means, covariance matrices, and mixing weights.
- Obtain a hard cluster label for each point using `predict()`, or the full soft responsibility matrix using `predict_proba()`.
- Identify uncertain points by inspecting the maximum responsibility per point, revealing where the model is genuinely unsure about cluster membership.
- Evaluate how well the model fits the data using the log-likelihood via `score()` and `score_samples()`.

But it is worth being precise about what you have actually built. A GMM is not simply a clustering algorithm that happens to use Gaussians. It is a **probabilistic density model**: it estimates the probability distribution that generated the data, and clustering is one thing you can do with it once it is fitted. The cluster labels produced by `predict()` are a byproduct of the model, not its primary output. The primary output is the fitted density $P(\mathbf{x})$, which tells you how probable any given point is under the model.

This distinction matters because a GMM can be used for things beyond clustering, a point we come back to in Section 8. Before that, though, there is a more immediate practical question to settle: how do you choose $K$, the number of components, in the first place? Everything so far has assumed $K$ was already known or given. In practice you almost never know it in advance, and getting it wrong in either direction, too few components or too many, changes how trustworthy the fitted model is. That is the problem the next section tackles.

---

## 7. Model Selection: BIC and AIC

🔙 [Back to Table of Contents](#table-of-contents)

### 7.1 The Problem with Choosing K

To fit a GMM, we must decide in advance how many components $K$ to use. This is not a question the EM algorithm can answer on its own: EM takes $K$ as given and finds the best parameters for that fixed number of components. Choosing $K$ is our responsibility.

The naive approach of simply trying many values of $K$ and picking the one with the highest log-likelihood does not work. As $K$ increases, the model gains more parameters and can always carve the data into finer subdivisions, so the log-likelihood never decreases. Given enough components, a GMM can fit any dataset arbitrarily well, including its noise. A model with $K$ equal to the number of data points would achieve a perfect log-likelihood by placing one tiny Gaussian on each point, which is clearly useless for understanding the structure of the data.

What we need is a way to penalise models for their complexity so that the score we optimise reflects not just how well the model fits the training data, but how well it is likely to generalise. Two criteria that do this are AIC and BIC.

### 7.2 AIC and BIC

Both the **Akaike Information Criterion (AIC)** and the **Bayesian Information Criterion (BIC)** follow the same structure: take the log-likelihood the model achieved, reverse its sign (so that lower is better, matching the convention of minimising a loss), and add a penalty that grows with the number of free parameters:

$$\text{AIC} = -2 \ell(\hat{\boldsymbol{\theta}}) + 2p$$

$$\text{BIC} = -2 \ell(\hat{\boldsymbol{\theta}}) + p \ln(n)$$

In both formulas, $\ell(\hat{\boldsymbol{\theta}})$ is the maximised log-likelihood, $p$ is the total number of free parameters in the model, and $n$ is the number of data points. **Lower values are better** for both criteria.

Reading each formula left to right: the first term $-2\ell$ decreases as the model fits the data better, which is good. The second term increases as the model uses more parameters, which is bad. The optimal $K$ is the one where these two forces are best balanced.

The key practical difference between the two is how aggressively they penalise complexity. AIC charges a fixed cost of 2 per free parameter regardless of how much data you have. BIC charges $\ln(n)$ per free parameter, which grows with dataset size. For any dataset with more than about 7 points $\ln(n) > 2$, so BIC is already stricter than AIC, and the gap widens as $n$ grows: with 1,000 data points the BIC penalty per parameter is $\ln(1000) \approx 6.9$, more than three times AIC's penalty of 2.

The consequence is that BIC tends to select a smaller $K$ than AIC. When the goal is to identify a genuine number of underlying groups in the data, BIC is generally the preferred criterion because its heavier penalty discourages splitting one real cluster into two just because doing so marginally improves the fit. AIC may be more appropriate when the goal is to build the best possible density model for prediction, where capturing fine structure in the data matters more than parsimony.

> 💡 **Intuition:** think of AIC and BIC as two judges scoring a model. Both reward a good fit to the data, but BIC is a stricter judge who becomes progressively harsher as the dataset grows larger: with more data, a genuinely better model should be able to prove itself without needing extra parameters.


---

#### BIC and AIC in Practice

scikit-learn computes both criteria directly from a fitted `GaussianMixture` object via `gmm.bic(X)` and `gmm.aic(X)`. Both methods take the data as an argument rather than just the model, because they need to evaluate the log-likelihood on that data to compute the score. The workflow is always the same: fit a GMM for each candidate value of $K$, record the BIC and AIC, and select the $K$ that minimises whichever criterion you have chosen.

```python
from sklearn.mixture import GaussianMixture

K_VALUES = range(1, 9)   # candidate values of K to evaluate

bic_scores = []
aic_scores = []

for k in K_VALUES:
    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        n_init=5,           # multiple restarts to avoid poor local optima
        random_state=0,
    )
    gmm.fit(X)
    bic_scores.append(gmm.bic(X))   # lower is better
    aic_scores.append(gmm.aic(X))   # lower is better

# The best K is the one that minimises the criterion. Adding 1 converts
# the zero-based list index back to the K value it corresponds to.
best_k_bic = bic_scores.index(min(bic_scores)) + 1
best_k_aic = aic_scores.index(min(aic_scores)) + 1

print(f'Best K by BIC: {best_k_bic}')
print(f'Best K by AIC: {best_k_aic}')
```

One practical detail: the loop above fits a separate GMM for each $K$ from scratch. This is necessary because `n_components` cannot be changed after a model has been fitted. Each call to `gmm.fit(X)` discards the previous model and replaces it with a new one, so only `bic_scores` and `aic_scores` are carried forward. If you need the fitted model for the best $K$ after this loop, you will need to refit it once more using the selected value.

The cell below extends this by fitting GMMs with $K = 1$ to $8$ components under both `'full'` and `'diag'` covariance structures and plotting BIC and AIC for both. Comparing the two covariance types on the same plot shows whether the additional parameters of a full covariance matrix are justified by a sufficiently better fit, or whether a simpler diagonal structure achieves a similar score with fewer parameters.

---

The code below fits GMMs with $K = 1$ to $8$ components on a fresh synthetic dataset where the true number of components is known to be 3. This ground-truth setup lets us check directly whether BIC and AIC recover the correct value of $K$, rather than just trusting that the minimum of the curve is meaningful.

Two covariance structures are compared for each value of $K$: `'full'`, which allows each component to take any elliptical shape, and `'diag'`, which constrains components to axis-aligned ellipses. Fitting both and plotting their scores on the same axes reveals whether the additional parameters of the full covariance model are earning their keep or whether the simpler diagonal structure achieves a comparable score with fewer parameters.

The printed output confirms that in this case both BIC and AIC agree and both correctly identify $K = 3$ as the optimal number of components. This is the ideal outcome, but it will not always happen in practice: on noisier data or with more overlapping clusters, the two criteria may disagree, or neither may land exactly on the true $K$. The plots are therefore as important as the printed minimum: a sharp, well-defined elbow in the curve gives more confidence in the selected $K$ than a broad, flat region where several values of $K$ produce similar scores.

In [ ]:
# Listing 9.
# ── Dataset ───────────────────────────────────────────────────────────────────
# A fresh three-component dataset with moderate overlap. We use make_blobs
# here so that the true K is known (3), giving us a ground truth to check
# whether BIC and AIC recover the correct number of components.
X_bic, _ = make_blobs(
    n_samples=400,
    centers=[[-4, -2], [1, 3], [5, 0]],
    cluster_std=1.5,
    random_state=0,
)

# ── Fit GMMs for K = 1 to 8 under two covariance structures ──────────────────
# For each K we fit two models: one with full covariance (most flexible, most
# parameters) and one with diagonal covariance (axis-aligned ellipses, fewer
# parameters). Comparing their BIC and AIC curves shows whether the extra
# parameters of the full covariance are justified by a better fit.
K_RANGE = range(1, 9)

bic_full, aic_full = [], []
bic_diag, aic_diag = [], []

for k in K_RANGE:
    g_full = GaussianMixture(
        n_components=k, covariance_type='full',
        n_init=5, random_state=0,
    )
    g_full.fit(X_bic)
    bic_full.append(g_full.bic(X_bic))
    aic_full.append(g_full.aic(X_bic))

    g_diag = GaussianMixture(
        n_components=k, covariance_type='diag',
        n_init=5, random_state=0,
    )
    g_diag.fit(X_bic)
    bic_diag.append(g_diag.bic(X_bic))
    aic_diag.append(g_diag.aic(X_bic))

# K values that minimise each criterion for the full covariance model.
x_ticks      = list(K_RANGE)
best_k_bic   = x_ticks[int(np.argmin(bic_full))]
best_k_aic   = x_ticks[int(np.argmin(aic_full))]
TRUE_K       = 3   # the number of components used to generate the data

print(f'BIC selects K = {best_k_bic} (full covariance)')
print(f'AIC selects K = {best_k_aic} (full covariance)')
print(f'True number of components: {TRUE_K}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# Left panel: BIC curves for both covariance types.
ax = axes[0]
ax.plot(x_ticks, bic_full, 'o-',  color='steelblue', lw=2,   ms=7, label='BIC (full)')
ax.plot(x_ticks, bic_diag, 's--', color='steelblue', lw=1.5, ms=6,
        alpha=0.6, label='BIC (diag)')
# Vertical line at the K that minimises BIC for the full covariance model.
ax.axvline(best_k_bic, color='steelblue', lw=1.5, linestyle=':',
           label=f'BIC minimum at K = {best_k_bic}')
# Vertical line at the true K so the reader can see whether BIC recovers it.
ax.axvline(TRUE_K, color='gray', lw=1.5, linestyle='--', alpha=0.5,
           label=f'True K = {TRUE_K}')
ax.set_xlabel('Number of components K')
ax.set_ylabel('BIC (lower = better)')
ax.set_title('BIC model selection\nmore conservative — prefers fewer components', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right panel: AIC curves for both covariance types.
ax = axes[1]
ax.plot(x_ticks, aic_full, 'o-',  color='tomato', lw=2,   ms=7, label='AIC (full)')
ax.plot(x_ticks, aic_diag, 's--', color='tomato', lw=1.5, ms=6,
        alpha=0.6, label='AIC (diag)')
ax.axvline(best_k_aic, color='tomato', lw=1.5, linestyle=':',
           label=f'AIC minimum at K = {best_k_aic}')
ax.axvline(TRUE_K, color='gray', lw=1.5, linestyle='--', alpha=0.5,
           label=f'True K = {TRUE_K}')
ax.set_xlabel('Number of components K')
ax.set_ylabel('AIC (lower = better)')
ax.set_title('AIC model selection\nless conservative — may prefer more components', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(
    'Figure 10: BIC and AIC for GMM model selection\n'
    'both penalise complexity; BIC imposes a heavier penalty for large n',
    fontsize=12,
)
plt.tight_layout()
plt.show()


---

## 8. GMM as a Density Estimator

🔙 [Back to Table of Contents](#table-of-contents)

So far we have treated the GMM primarily as a clustering tool: a way to identify sub-groups in data and assign points to those groups. But a GMM also provides something that K-Means fundamentally cannot: a **density model**. After fitting, the model encodes an explicit formula for $P(\mathbf{x})$ at any point in feature space, not just at the training points. You can query the model about any location and receive a number telling you how probable that location is under the learned distribution.

This is a qualitatively different kind of output from a cluster label. A cluster label tells you which group a point belongs to. A density value tells you how consistent a point is with the overall distribution the model has learned, regardless of which specific component it falls near. A point can receive a hard cluster label and still have a very low density if it sits in a sparsely populated region between two components.

This distinction opens up three important applications that go beyond clustering:

**Anomaly detection.** Any point that produces a very low value of $P(\mathbf{x})$ is improbable under the model. If new data arrives with extremely low density, it may be an outlier or anomaly: something the model has no good explanation for. A practical threshold is to flag points whose density falls below the bottom 2% of training densities, meaning the model considers them more improbable than 98% of the points it was trained on.

**Generative modelling.** Because a GMM is a full probabilistic model, you can draw new synthetic samples from it using `gmm.sample(n)`. Each generated point is statistically consistent with the training distribution. This is what we did in Section 5.3 when we sampled from a known GMM, and a fitted GMM can be used the same way.

**Out-of-distribution detection.** In a machine learning pipeline, you can fit a GMM to your training data and use it as a gatekeeper: before passing a new input to a classifier or predictor, check whether it falls within a region the model considers plausible. Inputs with very low density are flagged as out-of-distribution and treated with extra caution.

The key method for all of these is `score_samples`, which returns the log-density $\log P(\mathbf{x}_i)$ for each point rather than the raw probability $P(\mathbf{x}_i)$. Working in log-space is important because raw probabilities of individual points in high-dimensional space can be extremely small numbers that a computer cannot represent accurately. Log-densities stay in a numerically comfortable range and are directly comparable across points.

```python
# score_samples returns log P(x) for each input point.
log_density = gmm.score_samples(X)    # shape (n_samples,)

# To flag anomalies, set a threshold from the training data and mark
# any new point whose log-density falls below it.
threshold = np.percentile(gmm.score_samples(X_train), 2)   # bottom 2%
anomalies = log_density < threshold

# To generate new synthetic samples from the fitted distribution:
X_synthetic, component_labels = gmm.sample(100)   # 100 new points
```

The code below fits a GMM to `X_gmm`, the dataset generated in Section 5.3, and demonstrates both density estimation and anomaly detection. It evaluates $P(\mathbf{x})$ across a fine grid to produce a density map of the feature space, then introduces 40 new points drawn from a wide uniform distribution, most of which will fall in low-density regions far from the training data. Points whose log-density falls below the 2nd percentile of training densities are flagged as anomalies.



In [ ]:
# Listing 10.

# ── Fit a GMM to the dataset from Section 5.3 ────────────────────────────────
# X_gmm was generated from a known 3-component GMM. Fitting a fresh model
# here rather than reusing gmm_true means we are working with estimated
# parameters, as we would in a real application.
gmm_density = GaussianMixture(
    n_components=3, covariance_type='full',
    random_state=0, n_init=5,
)
gmm_density.fit(X_gmm)

# ── Evaluate log-density on a grid ───────────────────────────────────────────
# score_samples evaluates log P(x) at each grid point. Reshaping back to
# the grid shape allows contourf to render it as a continuous density map.
# np.exp converts from log-density to raw density P(x) for the colour scale.
h = 0.15
xlo, xhi = X_gmm[:, 0].min() - 1.5, X_gmm[:, 0].max() + 1.5
ylo, yhi = X_gmm[:, 1].min() - 1.5, X_gmm[:, 1].max() + 1.5
xx, yy   = np.meshgrid(np.arange(xlo, xhi, h), np.arange(ylo, yhi, h))
log_dens = gmm_density.score_samples(
    np.c_[xx.ravel(), yy.ravel()]
).reshape(xx.shape)

# ── Simulate anomalous points ─────────────────────────────────────────────────
# New points drawn uniformly across the full plotting region will mostly
# land in low-density areas far from the three training clusters, making
# them good candidates to be flagged as anomalies.
rng_anom   = np.random.default_rng(77)
X_anomaly  = rng_anom.uniform(low=[xlo, ylo], high=[xhi, yhi], size=(40, 2))
log_p_new  = gmm_density.score_samples(X_anomaly)

# The threshold is the 2nd percentile of the training log-density: any new
# point more improbable than 98% of training points is flagged.
threshold  = np.percentile(gmm_density.score_samples(X_gmm), 2)
flagged    = log_p_new < threshold

print(f'Anomaly threshold (2nd percentile of training log-density): {threshold:.3f}')
print(f'New points flagged as anomalies: {flagged.sum()} / {len(X_anomaly)}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# Left panel: the fitted density as a filled contour map.
# np.exp converts log-density to P(x) for the colour bar scale.
ax = axes[0]
cf = ax.contourf(xx, yy, np.exp(log_dens), levels=20, cmap='Blues', alpha=0.75)
plt.colorbar(cf, ax=ax, label='P(x) — estimated density')
ax.scatter(X_gmm[:, 0], X_gmm[:, 1],
           color='steelblue', s=10, alpha=0.4, edgecolors='none',
           label='Training data')
ax.set_title('Fitted GMM density\ncontours show P(x) across feature space', fontsize=10)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

# Right panel: anomaly detection. Normal new points are shown in green;
# flagged anomalies are shown as red crosses. The density contours are
# drawn in the background so the reader can see that flagged points
# consistently fall in low-density (lightly shaded) regions.
ax = axes[1]
ax.contourf(xx, yy, log_dens, levels=15, cmap='Blues', alpha=0.45)
ax.scatter(X_gmm[:, 0], X_gmm[:, 1],
           color='steelblue', s=10, alpha=0.3, edgecolors='none',
           label='Training data')
ax.scatter(X_anomaly[~flagged, 0], X_anomaly[~flagged, 1],
           color='seagreen', s=60, edgecolors='k', lw=0.8, zorder=5,
           label='New point — normal')
ax.scatter(X_anomaly[flagged, 0], X_anomaly[flagged, 1],
           color='tomato', s=90, edgecolors='k', lw=0.8,
           marker='X', zorder=6,
           label=f'Anomaly flagged (n={flagged.sum()})')
ax.set_title(
    f'Anomaly detection: flag points with log P(x) < {threshold:.2f}\n'
    f'(bottom 2% of training density)',
    fontsize=10,
)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

plt.suptitle(
    'Figure 11: GMM as a density estimator and anomaly detector',
    fontsize=12,
)
plt.tight_layout()
plt.show()

---

The printed output tells us that the GMM considers any point with a log-density below $-6.858$ to be anomalous, meaning it is more improbable under the fitted model than 98% of the training data. Of the 40 new points drawn uniformly across the feature space, 26 were flagged as anomalies, which is 65% of the new points.

This is a sensible result given how the new points were generated. Drawing uniformly across the full plotting region means most new points land in the large empty areas between and around the three clusters, where the training density is very low. The 14 points that were not flagged are the ones that happened to land close enough to one of the three clusters to receive a log-density above the threshold.

The two panels make this visible. In the left panel, the density map shows three bright peaks corresponding to the three fitted components, with density falling off rapidly in the surrounding space. The right panel overlays the new points on the same density background: the green points (not anomalous) sit visibly within or near the bright regions, while the red crosses (flagged) are scattered across the low-density areas in between and at the edges of the feature space.

Two things are worth noticing about the threshold itself. First, it is set entirely from the training data: we flag new points that are more improbable than the bottom 2% of training points, which means we expect roughly 2% of future normal points to be wrongly flagged as anomalies even if the data distribution has not changed. Second, the choice of 2% is arbitrary. A lower percentile produces a more lenient detector that only flags extreme outliers; a higher percentile flags more points but at the cost of more false alarms on normal data. Choosing the right threshold depends on the cost of missing a genuine anomaly versus the cost of incorrectly flagging a normal point, and there is no universally correct value.

---

Across Sections 5 to 8, you have built up a complete practical picture of Gaussian Mixture Models. It is worth pausing to summarise what you can now do and why each capability matters.

**Flexible clustering with uncertainty.** Unlike K-Means, a GMM does not force every point into a single cluster with equal confidence. The responsibility matrix from `predict_proba()` gives each point a probability of belonging to each component, and points in ambiguous regions between clusters receive split responsibilities rather than an arbitrary hard label. This is a more honest representation of the data when clusters overlap, and it lets you identify which assignments you should trust and which you should treat with caution.

* **Control over cluster shape.** The `covariance_type` parameter gives you direct control over what shapes your clusters can take, from circles (`'spherical'`) through axis-aligned ellipses (`'diag'`) to fully rotated ellipses in any orientation (`'full'`). K-Means implicitly assumes spherical clusters of equal size, which is a strong assumption that fails whenever clusters are elongated or correlated. A GMM with `covariance_type='full'` makes no such assumption and can adapt to whatever shape the data takes.

* **Principled selection of the number of components.** BIC and AIC give you a rigorous, quantitative way to choose $K$ rather than guessing or relying on visual inspection alone. By fitting models across a range of $K$ values and selecting the one that minimises BIC or AIC, you balance goodness of fit against model complexity and avoid overfitting. BIC is generally the more conservative choice and is preferred when you want to identify a genuine number of underlying groups. AIC may be more appropriate when your goal is predictive density rather than cluster discovery.

* **A density model, not just a clustering tool.** This is the most important distinction between a GMM and K-Means. A fitted GMM gives you an explicit formula for $P(\mathbf{x})$ at any point in feature space, not just at the training points. This means you can query the model about any new observation and receive a probability score reflecting how consistent it is with the learned distribution. No cluster algorithm that only produces labels can do this.

* **Anomaly detection.** The density model enables a straightforward and interpretable anomaly detector: flag any new point whose log-density falls below a threshold derived from the training data. This approach requires no additional modelling and no labelled anomalies to train on, making it a useful first-pass detector in any domain where you have a good characterisation of normal data but limited examples of what abnormal looks like.

* **Generative modelling.** A fitted GMM can generate new synthetic data points that are statistically consistent with the training distribution via `gmm.sample()`. This is useful for data augmentation, simulation, and stress-testing downstream models on plausible but unseen inputs.

Taken together, these capabilities make the GMM one of the most versatile unsupervised learning tools available. It is not always the right choice: if clusters are genuinely well-separated and spherical, K-Means is faster and simpler; if the data has very high dimensionality, the parameter count of a full covariance GMM can become difficult to estimate reliably. But when you need soft assignments, non-spherical clusters, a principled way to choose the number of clusters, or a density model that goes beyond cluster membership, the GMM is the natural starting point.


---

## 9. Strengths, Weaknesses, and When to Use GMMs

🔙 [Back to Table of Contents](#table-of-contents)

**Strengths:**

- **Soft (probabilistic) assignments.** Every point receives a probability of belonging to each component. This reflects genuine uncertainty rather than hiding it behind a false sense of certainty.
- **Flexible cluster shapes.** With full covariance, a GMM can model clusters of any elliptical shape and any orientation. K-Means, by contrast, implicitly assumes all clusters are spherical and of equal size.
- **Full density model.** The fitted GMM defines $P(\mathbf{x})$ everywhere in feature space, enabling generative sampling, density estimation, and anomaly detection, none of which K-Means supports.
- **Principled model selection.** BIC and AIC give a theoretically grounded way to choose the number of components, rather than relying on visual inspection of a plot.
- **Well-understood theory.** EM is guaranteed to converge to a local maximum, and its convergence properties are well characterised in the literature.

**Weaknesses:**

- **Slower than K-Means.** Each E-step and M-step involves matrix inversions and determinant computations for every component at every iteration. For large datasets with many features, this can be slow.
- **Sensitive to initialisation.** EM can converge to a poor local maximum. Mitigated by running multiple restarts (`n_init`), but this multiplies compute time.
- **Degeneracy risk.** If a component collapses onto a single data point, its covariance approaches zero and the likelihood blows up to infinity. scikit-learn adds a small regularisation term to the diagonal of each covariance matrix (controlled by `reg_covar`, default $10^{-6}$) to prevent this.
- **Struggles in high dimensions.** Estimating a full $d \times d$ covariance matrix requires, as a rough rule of thumb, at least $d(d+1)/2$ data points per component. In high dimensions, `'diag'` or `'spherical'` covariance is often more practical.
- **Assumes Gaussian components.** If the true sub-populations are non-Gaussian, for example heavy-tailed or themselves multi-modal, the GMM fit will be inaccurate. Non-parametric density estimators such as kernel density estimation may be more appropriate in those cases.

**When to use a GMM over K-Means:**

- When cluster boundaries are gradual or fuzzy and you want to quantify assignment uncertainty.
- When clusters are elongated or correlated rather than roughly spherical.
- When you need a generative model or want to evaluate $P(\mathbf{x})$ for new points.
- When you want a theoretically grounded criterion (BIC or AIC) for choosing the number of clusters.


---

## 10. Summary

🔙 [Back to Table of Contents](#table-of-contents)

**Mixture models and GMMs:**

| Concept | Key point |
|---------|-----------|
| Mixture model | Data arises from $K$ weighted component distributions; mixing weights $\pi_k$ satisfy $\sum_k \pi_k = 1$ |
| GMM | Gaussian components; each characterised by $\boldsymbol{\mu}_k$, $\boldsymbol{\Sigma}_k$, and $\pi_k$ |
| Latent variable | Component identity is unobserved; this is what makes direct MLE intractable |
| Hard vs soft | K-Means: binary assignments; GMM: probabilistic responsibilities $r_{ik} \in [0,1]$ |
| Responsibility | $r_{ik}$ = posterior probability that component $k$ generated point $i$ |

**The EM algorithm:**

| Step | What it does | Mathematical operation |
|------|-------------|----------------------|
| E-step | Compute responsibilities given current parameters | $r_{ik} \propto \pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ |
| M-step | Update parameters given responsibilities | Weighted mean, weighted scatter, normalised weights |
| Convergence | Guaranteed to increase log-likelihood; stops when change $< $ tolerance | Local, not global maximum |

**Covariance types (sklearn):**

| Type | Shape | Flexibility | Parameters |
|------|-------|-------------|-----------|
| `'spherical'` | $\sigma_k^2 \mathbf{I}$ | Lowest | 1 per component |
| `'diag'` | Diagonal | Medium | $d$ per component |
| `'tied'` | Full, shared | Medium | $d(d+1)/2$ total |
| `'full'` | Full, per component | Highest | $d(d+1)/2$ per component |

**Model selection:**

| Criterion | Penalty | Tendency |
|-----------|---------|---------|
| AIC | $2p$ | More liberal; may over-estimate $K$ |
| BIC | $p \ln n$ | More conservative; generally preferred for cluster discovery |


---

## 11. References

🔙 [Back to Table of Contents](#table-of-contents)

Bishop, C. M. (2006). *Pattern Recognition and Machine Learning* (Chapter 9: Mixture Models and EM). Springer.

Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society, Series B*, 39(1), 1--38.

McLachlan, G. J., & Peel, D. (2000). *Finite Mixture Models*. John Wiley & Sons.

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825--2830.

Schwarz, G. (1978). Estimating the dimension of a model. *Annals of Statistics*, 6(2), 461--464.

Virtanen, P., Gommers, R., Oliphant, T. E., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods*, 17(3), 261--272. https://doi.org/10.1038/s41592-019-0686-2